# Kubernetes cluster

## Prerequisites

Before starting, make sure you:

- Have completed the [Preparation](./00_preparation.ipynb) and the [Provisioning resources](./01_provision.ipynb) tutorial.
- Have a basic understanding of Python.
- You have basic kubernetes knowledge

## Introduction

The orchestrator relies on Kubernetes to deploy most of its network functions. An essential step in the workflow is creating a Kubernetes cluster composed of all nodes defined in the deployment model (specifically, those flagged as `is_provisioned: true`).

To accomplish this, you use the `K8SManagement` manager, whose role is to deploy and configure a Kubernetes cluster compatible with your desired 5G network deployment.

In this tutorial, the cluster will consist of `node1`, `node2`, and `node3`, which you provisioned in the [Provisioning Resources tutorial](./provision.ipynb).

### Node Labeling and Scheduling

Nodes are labeled to ensure workloads are scheduled correctly (for example, ensuring UEs are only deployed on designated worker nodes). Internally, the library applies the `node-type.5g.blueprint` label to nodes in the cluster. For instance, if a node supports the `ran` role, it receives the label `node-type.5g.blueprint/ran=true`.

### Model Continuity Across Notebooks

As mentioned earlier, managers consume a metamodel and a deployment model, utilizing the output from the previous step. When instantiating the Kubernetes manager, you simply specify its preceding manager (in this case, `prov`), and it will automatically retrieve the required models.

However, because Jupyter notebooks do not share in-memory variables across files, you cannot directly access the `prov` instance from the provisioning notebook. Fortunately, because you saved the metamodel and deployment model to `state/provision/metamodel.prov.yaml` and `state/provision/deployment_model.prov.yaml`, you can easily instantiate the reference state as follows:

In [1]:
from orchestrator.deployment import ProvisioningManagement

PROV_DIR="state/provision"
METAMODEL_FILE = f"{PROV_DIR}/metamodel.prov.yaml"
DEPLOYMENT_MODEL_FILE = f"{PROV_DIR}/deployment_model.prov.yaml"

prov = ProvisioningManagement(metamodel_file=METAMODEL_FILE, deployment_model_file=DEPLOYMENT_MODEL_FILE)

> **Important:** The orchestrator library follows a declarative and functional design. Consequently, metamodels and deployment models are duplicated when passed between managers, ensuring that input models are never mutated in place.

Once the manager is properly instantiated, you trigger the cluster deployment. This process builds a fully functioning Kubernetes cluster across the provisioned nodes and automatically distributes the corresponding `kubeconfig` file to all machines to streamline subsequent administrative tasks.

> **Note:** To onboard existing Kubernetes clusters rather than provisioning new ones, refer to the [orchestrator library documentation](https://gitlab.inria.fr/slices-ri/blueprints/post-5g/orchestrator).

In [2]:
from orchestrator.deployment import K8SManagement

k8s = K8SManagement(metamodel_file=METAMODEL_FILE, deployment_model_file=DEPLOYMENT_MODEL_FILE)


In [3]:
await k8s.deploy(quiet=True)            # don't show output

Once it is deployed we can verify that the kubernets cluster is deployed and 
setup correctly by running the `kubectl` command.

In [4]:
print((await prov.run_node_command(
    command=["kubectl", "get", "nodes", "-o", "wide", "--show-labels"],
    nodes=["node1"],
    become=True, quiet=True)))

NAME    STATUS   ROLES           AGE   VERSION   INTERNAL-IP     EXTERNAL-IP   OS-IMAGE             KERNEL-VERSION             CONTAINER-RUNTIME    LABELS
node1   Ready    control-plane   98s   v1.36.2   10.10.222.93    <none>        Ubuntu 24.04.3 LTS   6.8.0-85-generic (amd64)   containerd://2.2.1   beta.kubernetes.io/arch=amd64,beta.kubernetes.io/os=linux,kubernetes.io/arch=amd64,kubernetes.io/hostname=node1,kubernetes.io/os=linux,node-role.kubernetes.io/control-plane=,node-type.5g.blueprint/core=true,node.kubernetes.io/exclude-from-external-load-balancers=
node2   Ready    <none>          67s   v1.36.2   10.10.218.199   <none>        Ubuntu 24.04.3 LTS   6.8.0-85-generic (amd64)   containerd://2.2.1   beta.kubernetes.io/arch=amd64,beta.kubernetes.io/os=linux,kubernetes.io/arch=amd64,kubernetes.io/hostname=node2,kubernetes.io/os=linux,node-type.5g.blueprint/ran=true
node3   Ready    <none>          67s   v1.36.2   10.10.219.111   <none>        Ubuntu 24.04.3 LTS   6.8.0-85-generic (

In [5]:
print((await prov.run_node_command(
    command=["kubectl", "get", "all", "-A", "-o", "wide"],
    nodes=["node1"],
    become=True, quiet=True)))

NAMESPACE         NAME                                           READY   STATUS    RESTARTS      AGE    IP               NODE    NOMINATED NODE   READINESS GATES
calico-system     pod/calico-kube-controllers-58c6d6b565-hdtdh   1/1     Running   0             83s    10.244.166.130   node1   <none>           <none>
calico-system     pod/calico-node-fvr76                          1/1     Running   0             75s    10.10.219.111    node3   <none>           <none>
calico-system     pod/calico-node-mz8xl                          1/1     Running   0             75s    10.10.218.199    node2   <none>           <none>
calico-system     pod/calico-node-sbknq                          1/1     Running   0             76s    10.10.222.93     node1   <none>           <none>
calico-system     pod/calico-typha-58b56b46c6-cx6tk              1/1     Running   0             83s    10.10.222.93     node1   <none>           <none>
calico-system     pod/calico-typha-58b56b46c6-m2dfm              1/1     

Cluster information is in kept in the deployment model in its `k8s` section of `params`. 

In [6]:
print(k8s.deployment_model.params.k8s.to_yaml(exclude_defaults=False))


timezone: Europe/Paris
k8s_version:
  major: 1
  minor: 36
  patch: 2
  extension: '2.1'
cluster_name: 5gbp
master_node: node1
join_command: 'kubeadm join 10.10.222.93:6443 --token y2qgag.igv745kd40y10n55 --discovery-token-ca-cert-hash
  sha256:66d89ee6b3a011e5043656b2f8abb587514f2f74e294d2fd02bb1223c4a745b1 '
pod_network_cidr: 10.244.0.0/16
service_cidr: 10.96.0.0/12
calico_version: 3.32.1
install_multus: true
multus_version: 4.3.0
helm_version: 4.2.3
k9s_install: true
k9s_version: 0.51.0
containerd_snapshotter: null
enable_cni_dhcp: true
copy_kubeconfig_to_workers: true
local_kubeconfig_path: /workspaces/orchestrator/.venv-1/lib/python3.12/.k8s-admin.conf
label_selection_base: node-type.5g.blueprint
used_ips: []



If you wish to access the cluster manually using `kubectl`, the local path to the generated cluster configuration is available in the `local_kubeconfig_path` attribute (accessible via `k8s.deployment_model.params.k8s.local_kubeconfig_path`). This file is automatically fetched and copied from the master node to your local machine during cluster deployment.

In [7]:
local_k8s_path = k8s.deployment_model.params.k8s.local_kubeconfig_path
print (local_k8s_path)

/workspaces/orchestrator/.venv-1/lib/python3.12/.k8s-admin.conf


The orchestrator does not interact with the cluster directly from your local machine. Instead, it establishes an SSH connection to the master node and executes all operations directly from there.

If you need to access the cluster API server from your local machine, it may not be directly reachable depending on your network topology. While a detailed discussion of routing options is outside the scope of this tutorial, a straightforward solution is to use SSH port forwarding. 

Connection details for each node can be found in the `ssh_info` section of the deployment model. You can establish port forwarding using a command similar to the following:

```bash
ssh -L 6443:localhost:6443 -J jump_to_reach_the_machine login@host
```

Once the tunnel is open, update the `cluster.server` URL in your local kubeconfig file to point to `https://localhost:6443` instead of the internal cluster endpoint.

In [8]:
print (k8s.deployment_model.nodes["node1"].ssh_info.to_yaml())

hostname: 10.10.222.93
user: ubuntu
jumphost:
  hostname: bastion2.slices-be.eu
  user: proxy
  common_args: -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o ServerAliveInterval=30
    -o ServerAliveCountMax=3 -o ControlMaster=auto -o ControlPersist=7200s -o ControlPath=/tmp/cm-%C
common_args: -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o ServerAliveInterval=30
  -o ServerAliveCountMax=3 -o ControlMaster=auto -o ControlPersist=7200s -o ControlPath=/tmp/cm-%C
  -o ProxyCommand="ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null
  -W %h:%p proxy@bastion2.slices-be.eu -p 22"



In [9]:
!cat {local_k8s_path} | sed -E '/^[[:space:]]*(cluster|client-certificate-data|client-key-data|certificate-authority-data):/s/:.*/: <REDACTED>/'

apiVersion: v1
clusters:
- cluster:
    certificate-authority-data: <REDACTED>
    server: https://10.10.222.93:6443
  name: 5gbp
contexts:
- context:
    cluster: <REDACTED>
    user: kubernetes-admin
  name: kubernetes-admin@5gbp
current-context: kubernetes-admin@5gbp
kind: Config
users:
- name: kubernetes-admin
  user:
    client-certificate-data: <REDACTED>
    client-key-data: <REDACTED>


> **Note:** For reproducibility and operational consistency, we strongly recommend using the orchestrator library itself to interact with the cluster. The library provides native support for executing commands directly on cluster nodes, allowing you to run `kubectl` or `helm` commands natively as demonstrated above for example.

```python
print((await prov.run_node_command(
    command=["kubectl", "get", "all", "-A", "-o", "wide"],
    nodes=["node1"],
    become=True, quiet=True)))
```